# Import

In [ ]:
# Import
%load_ext autoreload
%autoreload 2
import re
import os
import pandas as pd
from pathlib import Path
import sys
import json

# Automation

In [ ]:
selected_lemmas = ['attack_nn', 'bag_nn', 'ball_nn', 'bit_nn', 'chairman_nn', 'circle_vb', 'contemplation_nn', 'donkey_nn', 'edge_nn', 'face_nn', 'fiction_nn', 'gas_nn', 'graft_nn', 'head_nn', 'land_nn', 'lane_nn', 'lass_nn', 'multitude_nn', 'ounce_nn', 'part_nn', 'pin_vb', 'plane_nn', 'player_nn', 'prop_nn', 'quilt_nn', 'rag_nn', 'record_nn', 'relationship_nn', 'risk_nn', 'savage_nn', 'stab_nn', 'stroke_vb', 'thump_nn', 'tip_vb', 'tree_nn', 'twist_nn', 'word_nn']

In [ ]:
# Automate all the steps for all selected lemmas

# Specify required input
# Add SynFlow to path in order to import modules
repo_root = "../"
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

results = {}

# Specify corpus and output folders
period = '1-2'
corpus_folder = f'../SemEval_en_no_split/merged_corpus/'
output_folder = Path(f'../case_studies/SemEval_en_no_split')

for selected_lemma in selected_lemmas:
    # Specify target lemma and part of speech
    target_lemma, selected_pos = selected_lemma.split('_')

    if selected_pos == 'nn':
        target_pos = 'N'
    elif selected_pos == 'vb':
        target_pos = 'V'
    else:
        raise ValueError(f'Unknown part of speech: {selected_pos}')

    # Input target
    keyword_string = f'{target_lemma}\t{target_pos}' # Or you can use the full POS for precision (e.g., {target_lemma}\tNOUN)

    # Pattern of the file names and regex patterns of the CONLLU file
    fname_pattern = re.compile(
        r'^ccoha[12]_reparsed\.txt$'
    )
    corpus_pattern = re.compile(
        r'([^\t]+)\t'      # word form
        r'([^\t]+)\t'      # lemma
        r'([^\t])[^\t]*\t' # POS-init (UPOS or XPOS) or r'([^\t]+)\t'      # FULL POS 
        r'([^\t]+)\t'      # ID
        r'([^\t]+)\t'      # HEAD
        r'([^\t]+)'        # DEPREL
    )

    # The path to the slot count JSON file
    slot_json_path = f'../case_studies/SemEval_en_no_split/output/{target_lemma}-{target_pos}-1-2/Explorer/{target_lemma}_{target_pos}_spaths.json'

    # Dont change below this line
    output_folder_lemma = output_folder / 'output' / f'{target_lemma}-{target_pos}-{period}'
    output_explorer = f'{output_folder_lemma}/Explorer'
    output_embedding = f'{output_folder_lemma}/Embedding'
    input_SCD = output_folder / 'input' / 'SCD' /f'{target_lemma}-{target_pos}-{period}'

    os.makedirs(output_explorer, exist_ok=True)
    os.makedirs(output_embedding, exist_ok=True)
    os.makedirs(input_SCD, exist_ok=True)

    # Slot Frequencies
    # Token counts for normalisation
    from SynFlow.SCD import count_keyword_tokens_by_period
    token_counts = count_keyword_tokens_by_period(corpus_folder, keyword_string,
                                                fname_pattern=fname_pattern)
    print(token_counts)

    from SynFlow.SCD import plot_freq_top_union_slots_by_period

    plot_freq_top_union_slots_by_period(
        json_path=slot_json_path,
        top_n=10,
        normalized=False,
        relative=False,
        token_counts=token_counts,
    )

    from SynFlow.SCD import freq_all_slots_by_period
    slot_raw_freq_df = freq_all_slots_by_period(json_path=slot_json_path).T
    slot_raw_freq_df.head(30)

    # Get all slots in the corect format
    from SynFlow.Explorer.sfiller_df import get_all_slots
    all_slots = get_all_slots(slot_raw_freq_df)

    # Building a slot filler df
    from SynFlow.Explorer import build_sfiller_df

    df_slots = build_sfiller_df(
        corpus_folder=corpus_folder,
        template=all_slots, 
        target_lemma=target_lemma,
        target_pos=target_pos,
        pattern=corpus_pattern,
        # freq_path='/home/volt/bach/pilot_data/RSC/lemma_pos_init_freq.txt', # Be sure that the freq_path matches that of the filter format
        # freq_min=1,
        # freq_max=100_000_000,
        filtered_pos=[],
        filler_format='lemma/pos', # lemma/deprel or 'lemma/pos'
        output_folder= output_explorer
        )
    
    all_sfillers_csv_path = f'../case_studies/SemEval_en_no_split/output/{target_lemma}-{target_pos}-1-2/Explorer/{target_lemma}_samples_sfillerdf_all.csv'

    # Calculate divergences of all slots
    from SynFlow.SCD import consecutive_JSD_dict
    sfiller_df_path = all_sfillers_csv_path

    consecutive_JSD_dictionary = consecutive_JSD_dict(all_sfillers_csv_path=sfiller_df_path,
                     min_freq=1,
                     mode='all' # or 'data_only' if you want to skip the empty periods
                     )
    
    # Store results
    results[selected_lemma] = consecutive_JSD_dictionary

# Benchmarking 

In [ ]:
# Raw results
results

In [ ]:
# Transform results
# Remove slots with empty or 0 JSD scores

def transform_results(results):
    transformed_results = {}

    for lemma, slot_per_jsd in results.items():
        transformed_results[lemma] = {}
        for slot, per_jsd in slot_per_jsd.items():
            jsd = next(iter(per_jsd.values()), None)
            if jsd is None:
                continue
            elif jsd > 0:
                transformed_results[lemma][slot] = jsd

    return transformed_results

In [ ]:
transformed_results = transform_results(results)
transformed_results

In [ ]:
# Dump results
with open("./result_semeval.json", "w", encoding="utf-8") as f:
    json.dump(transformed_results, f, ensure_ascii=False, indent=2, sort_keys=True)

In [ ]:
# Read results
with open("./result_semeval.json", "r", encoding="utf-8") as f:
    transformed_results = json.load(f)

In [ ]:
gold_file_task1 = '../SemEval/semeval2020_ulscd_eng/truth/binary.txt'
gold_file_task2 = '../SemEval/semeval2020_ulscd_eng/truth/graded.txt'

with open(gold_file_task2, 'r') as f:
    gold_task2 = f.read().splitlines()
    gold_task2 = [g.split('\t') for g in gold_task2]
    gold_task2 = {g[0]: float(g[1]) for g in gold_task2}

with open(gold_file_task1, 'r') as f:
    gold_task1 = f.read().splitlines()
    gold_task1 = [g.split('\t') for g in gold_task1]
    gold_task1 = {g[0]: int(g[1]) for g in gold_task1}

results_summary = {}
threshs = [0.5]

for selected_lemma in selected_lemmas:
    
    # create a list of JSDs of all slots of the lemmas
    JSD_scores = [slot_JSD
              for slot_JSD in transformed_results.get(selected_lemma, {}).values()
              ]

    # Max slot JSD of a lemma
    max_score = max(JSD_scores) if JSD_scores else 0

    sum_scores = {}
    mean_scores = {}

    for thresh in threshs:
        scores_abv_thresh = [sat for sat in JSD_scores if sat > thresh]

        sum_scores[thresh] = sum(scores_abv_thresh)
        mean_scores[thresh] = (sum_scores[thresh] / len(scores_abv_thresh)) if scores_abv_thresh else 0

    results_summary[selected_lemma] = {
        'gold_task1': gold_task1.get(selected_lemma, None),
        'gold_task2': gold_task2.get(selected_lemma, None),
        'max_JSD': max_score,
        **{f'mean_JSD_{thresh}': mean_scores[thresh] for thresh in threshs},
        **{f'sum_JSD_{thresh}': sum_scores[thresh] for thresh in threshs}
    }


In [ ]:
results_summary

In [ ]:
# Benchmark with the 2 shared task
import pandas as pd

# results_summary: dict {lemma -> {metric_name -> value}}
# Create a dataframe and csv
result_df = pd.DataFrame.from_dict(results_summary, orient='index')
result_df.to_csv('./semeval_slot_level_summary.csv')

result_df = result_df.apply(pd.to_numeric, errors='coerce')

# Spearman with gold_task2
corr = result_df.corr(method='spearman')
gold_corr = corr.loc['gold_task2'].drop([c for c in ['gold_task2', 'gold_task1'] if c in corr.columns])
print("Spearman vs gold_task2:")
print(gold_corr.sort_values(ascending=False))

# Assign 1 for the top 43% for each metric cols and 0 for the rest và compute accuracy for gold_task1
gold_task1 = result_df['gold_task1'].astype(int)
metric_cols = [c for c in result_df.columns if c not in ('gold_task1', 'gold_task2')]

acc_results = {}
top_pct = 0.43
for col in metric_cols:
    col_vals = result_df[col].dropna()
    if col_vals.empty:
        continue
    
    # item in top pct
    k = int(len(col_vals) * top_pct)
    if k < 1:
        k = 1

    # sort
    sorted_vals = col_vals.sort_values(ascending=False)
    threshold = sorted_vals.iloc[k - 1]

    # assign 1 if >= threshold, 0 if < threshold (NaN -> 0)
    preds = (result_df[col] >= threshold).fillna(False).astype(int)

    # Accuracy wwith gold_task1
    acc = (preds == gold_task1).mean()
    acc_results[col] = acc

acc_series = pd.Series(acc_results).sort_values(ascending=False)
print(f"\nTask 1 accuracy (top {top_pct} thresholding):")
print(acc_series)

# Remove the POS of the context lemmas

In [ ]:
# Helpers
import ast
from pathlib import Path

import pandas as pd


def _strip_pos_in_list_str(cell: str) -> str:
    """
    Take a string like "['attack/N', 'require/V']" and return "['attack', 'require']".
    If the string is not a list repr, return it unchanged.
    """
    if cell is None:
        return cell

    if not isinstance(cell, str):
        return cell

    s = cell.strip()
    # Only touch things that look like a list
    if not (s.startswith("[") and s.endswith("]")):
        return cell

    try:
        parsed = ast.literal_eval(s)
    except Exception:
        # If it doesn't parse as Python literal, leave unchanged
        return cell

    if not isinstance(parsed, list):
        return cell

    new_items = []
    for item in parsed:
        if isinstance(item, str):
            # Remove POS: take everything before the last '/'
            if "/" in item:
                word = item.rsplit("/", 1)[0]
                new_items.append(word)
            else:
                new_items.append(item)
        else:
            new_items.append(item)

    # Convert back to a list representation string
    return repr(new_items)


def remove_pos_from_csv(input_path: str, output_path: str | None = None) -> str:
    """
    Input: path to CSV file.
    Output: path to new CSV file with POS removed inside bracketed lists.
    """
    input_p = Path(input_path)
    if output_path is None:
        output_p = input_p.with_name(input_p.stem + "_nopos" + input_p.suffix)
    else:
        output_p = Path(output_path)

    # Read everything as string to avoid type guessing
    df = pd.read_csv(input_p, dtype=str)

    # Apply to every column (including 'target' etc.)
    for col in df.columns:
        df[col] = df[col].apply(_strip_pos_in_list_str)

    df.to_csv(output_p, index=False)
    return str(output_p)

In [ ]:
results = {}
for selected_lemma in selected_lemmas:
    # Specify target lemma and part of speech
    target_lemma, selected_pos = selected_lemma.split('_')

    if selected_pos == 'nn':
        target_pos = 'N'
    elif selected_pos == 'vb':
        target_pos = 'V'
    else:
        raise ValueError(f'Unknown part of speech: {selected_pos}')
      
    all_sfillers_csv_path_org = f'../case_studies/SemEval_en_no_split/output/{target_lemma}-{target_pos}-1-2/Explorer/{target_lemma}_samples_sfillerdf_all.csv'
    all_sfillers_csv_path_no_POS = f'../case_studies/SemEval_en_no_split/output/{target_lemma}-{target_pos}-1-2/Explorer/{target_lemma}_samples_sfillerdf_all_no_POS.csv'
    remove_pos_from_csv(all_sfillers_csv_path_org, all_sfillers_csv_path_no_POS)

    # Calculate divergences of all slots
    from SynFlow.SCD import consecutive_JSD_dict
    # Select the file to process, POS or no POS
    # sfiller_df_path = all_sfillers_csv_path_org
    sfiller_df_path = all_sfillers_csv_path_no_POS
    
    consecutive_JSD_dictionary = consecutive_JSD_dict(all_sfillers_csv_path=sfiller_df_path,
                     min_freq=2,
                     mode='all' # or 'data_only' if you want to skip the empty periods
                     )
    
    # Store results
    results[selected_lemma] = consecutive_JSD_dictionary

In [ ]:
transformed_results = transform_results(results)
transformed_results

In [ ]:
# Dump results
with open("./result_semeval_no_POS.json", "w", encoding="utf-8") as f:
    json.dump(transformed_results, f, ensure_ascii=False, indent=2, sort_keys=True)

# Read results
with open("./result_semeval_no_POS.json", "r", encoding="utf-8") as f:
    transformed_results = json.load(f)

In [ ]:
gold_file_task1 = '../SemEval/semeval2020_ulscd_eng/truth/binary.txt'
gold_file_task2 = '../SemEval/semeval2020_ulscd_eng/truth/graded.txt'

with open(gold_file_task2, 'r') as f:
    gold_task2 = f.read().splitlines()
    gold_task2 = [g.split('\t') for g in gold_task2]
    gold_task2 = {g[0]: float(g[1]) for g in gold_task2}

with open(gold_file_task1, 'r') as f:
    gold_task1 = f.read().splitlines()
    gold_task1 = [g.split('\t') for g in gold_task1]
    gold_task1 = {g[0]: int(g[1]) for g in gold_task1}

results_summary = {}
threshs = [0.5]

for selected_lemma in selected_lemmas:
    
    # create a list of JSDs of all slots of the lemmas
    JSD_scores = [slot_JSD
              for slot_JSD in transformed_results.get(selected_lemma, {}).values()
              ]

    max_score = max(JSD_scores) if JSD_scores else 0
    sum_scores = {}
    mean_scores = {}

    for thresh in threshs:
        scores_abv_thresh = [sat for sat in JSD_scores if sat > thresh]

        sum_scores[thresh] = sum(scores_abv_thresh)
        mean_scores[thresh] = (sum_scores[thresh] / len(scores_abv_thresh)) if scores_abv_thresh else 0

    results_summary[selected_lemma] = {
        'gold_task1': gold_task1.get(selected_lemma, None),
        'gold_task2': gold_task2.get(selected_lemma, None),
        'max_JSD': max_score,
        **{f'mean_JSD_{thresh}': mean_scores[thresh] for thresh in threshs},
        **{f'sum_JSD_{thresh}': sum_scores[thresh] for thresh in threshs}
    }


In [ ]:
# Benchmark with the 2 shared task
import pandas as pd

# results_summary: dict {lemma -> {metric_name -> value}}
# DataFrame and CSV
result_df = pd.DataFrame.from_dict(results_summary, orient='index')
result_df.to_csv('./semeval_slot_level_summary.csv')

result_df = result_df.apply(pd.to_numeric, errors='coerce')

# Spearman with gold_task2
corr = result_df.corr(method='spearman')
gold_corr = corr.loc['gold_task2'].drop([c for c in ['gold_task2', 'gold_task1'] if c in corr.columns])
print("Spearman vs gold_task2:")
print(gold_corr.sort_values(ascending=False))

# 4) Assign 1 for the top 43% for each metric cols and 0 for the rest và compute accuracy for gold_task1
gold_task1 = result_df['gold_task1'].astype(int)
metric_cols = [c for c in result_df.columns if c not in ('gold_task1', 'gold_task2')]

acc_results = {}
top_pct = 0.43
for col in metric_cols:
    col_vals = result_df[col].dropna()
    if col_vals.empty:
        continue
    
    # items in top top_pct%
    k = int(len(col_vals) * top_pct)
    if k < 1:
        k = 1

    sorted_vals = col_vals.sort_values(ascending=False)
    threshold = sorted_vals.iloc[k - 1]

    # 1 if >= threshold, 0 if < threshold (NaN -> 0)
    preds = (result_df[col] >= threshold).fillna(False).astype(int)

    # Accuracy with gold_task1
    acc = (preds == gold_task1).mean()
    acc_results[col] = acc

acc_series = pd.Series(acc_results).sort_values(ascending=False)
print(f"\nTask 1 accuracy (top {top_pct} thresholding):")
print(acc_series)